# Day 10 — HOL 1: Implement SCD Type 2 for `dim_customer` Using MERGE

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 1 — SCD Type 1 & Type 2 using MERGE |
| **Source** | `gbmart.bronze.customers`, `gbmart.silver.customers` (read-only — you clone them below) |
| **Duration** | 60 minutes |
| **Output** | Your own scratch-schema `silver_customers` table showing a real SCD2 version history |

### Learning Objectives
- Implement the real 6-step SCD2 MERGE pattern GlobalMart uses for `dim_customer`
- Read only the changed rows via Change Data Feed, never a full table re-scan
- Prove to yourself that a changed customer produces exactly 2 Silver rows: an old, closed-out version and a new, current one

---
**Why you're working in your own schema, not the shared `gbmart` tables:** an SCD2 MERGE consumes whatever pending change exists the first time it runs. If everyone in the class ran this against the real `gbmart.silver.customers`, only the first person to run it would see anything happen — everyone after would find no changes left to merge. Working in your own clone means your hands-on is fully yours, repeatable as many times as you like.

**Instructions:** Replace `YOUR_SCHEMA` in the setup cell with something unique to you (e.g. `main.priya_scd_lab`). Run each cell in order with **Shift + Enter**.

## Setup — Clone Your Own Copy

`SHALLOW CLONE` gives you an independent table (your own transaction log, your own history) without copying the underlying data files — instant, and safe to write to.

In [ ]:
# ─── Replace YOUR_SCHEMA with something unique to you, e.g. main.priya_scd_lab ───
YOUR_SCHEMA = "main.YOUR_SCHEMA"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_SCHEMA}")

BRONZE_TABLE = f"{YOUR_SCHEMA}.bronze_customers"
SILVER_TABLE = f"{YOUR_SCHEMA}.silver_customers"

spark.sql(f"CREATE OR REPLACE TABLE {BRONZE_TABLE} SHALLOW CLONE gbmart.bronze.customers")
spark.sql(f"CREATE OR REPLACE TABLE {SILVER_TABLE} SHALLOW CLONE gbmart.silver.customers")

# CDF is a table property, not data — cloning doesn't carry it over, so enable it fresh
spark.sql(f"ALTER TABLE {BRONZE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Cloned into {YOUR_SCHEMA}: bronze_customers, silver_customers (CDF enabled on bronze)")

## Step 1 — Read All Data (Current Full State of Bronze)

Not for processing yet — just to see where things stand before isolating the incremental piece.

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

bronze_df = spark.table(BRONZE_TABLE)
print(f"Total rows currently in Bronze: {bronze_df.count():,}")

## Step 2 — Find the Starting Point for CDF

`DESCRIBE HISTORY` shows every write to this table as a version. We need the version number **right before** the incremental change we're about to make — CDF will read everything **after** it.

In [ ]:
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}") \
    .select("version", "timestamp", "operation") \
    .orderBy("version") \
    .display()

In [ ]:
# Set this from the history output above — the version right before your change
LAST_PROCESSED_VERSION = spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").selectExpr("max(version)").collect()[0][0]
print(f"LAST_PROCESSED_VERSION = {LAST_PROCESSED_VERSION}")

## Simulate an Incoming Change

In production, this Bronze table gets updated by the normal Autoloader ingestion path when a new file lands. For this hands-on, you'll apply a change directly to your clone so there's a real, fresh commit for CDF to detect — pick any 2 real customers and give them a new email address.

> **Note:** if a `customer_consent_*.csv`-style file has ever landed in the same source folder as your customers data, its rows would show up in Bronze history (and in the CDF read below) too — that's a different entity, not part of this customer-attribute SCD2 flow. If you see unexpected rows in Step 3, filter by requiring the core customer columns to be non-null, as Step 4 already does.

In [ ]:
# YOUR CODE HERE — pick 2 CustomerIDs from your Bronze clone and UPDATE their Email
# Hint: SELECT 2 CustomerID values, then run an UPDATE ... SET Email = ... WHERE CustomerID = ... for each

sample_ids = [row.CustomerID for row in spark.table(BRONZE_TABLE).select("CustomerID").limit(2).collect()]

for i, cid in enumerate(sample_ids):
    spark.sql(f"""
        UPDATE {BRONZE_TABLE}
        SET Email = 'new.email.{i}@globalmart-hol.com'
        WHERE CustomerID = '{cid}'
    """)

print(f"Changed emails for: {sample_ids}")

## Step 3 — Take Only the Changes (CDF, Not a Full Re-Scan)

`readChangeFeed` + `startingVersion` returns only the rows that changed since `LAST_PROCESSED_VERSION` — not the whole table. `update_preimage` (the "before" snapshot) is dropped; we only want `insert` and `update_postimage` (the "after" state).

In [ ]:
cdf_changes_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(BRONZE_TABLE)
        .filter("_change_type != 'update_preimage'")
)

print(f"Changed rows via CDF: {cdf_changes_df.count():,}")
cdf_changes_df.select("CustomerID", "Email", "_change_type", "_commit_version").display()

**Q: How many rows did CDF return, and what `_change_type` values do you see? Why is `update_preimage` filtered out?**

*Your answer:* _______________

## Step 4 — Process Only These Changed Rows

Same cleaning logic Day 5 used for the full load (`full_name` from `FirstName`+`LastName`, column renames to snake_case) — just scoped to the small CDF result instead of the whole table.

In [ ]:
processed_df = cdf_changes_df \
    .filter(col("CustomerID").isNotNull() & col("Email").isNotNull()) \
    .withColumn("FirstName", trim(col("FirstName"))) \
    .withColumn("LastName", trim(col("LastName"))) \
    .withColumn("full_name", concat_ws(" ", col("FirstName"), col("LastName"))) \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("Email", "email") \
    .withColumnRenamed("PhoneNumber", "phone_number") \
    .withColumnRenamed("DateOfBirth", "date_of_birth") \
    .withColumnRenamed("RegistrationDate", "registration_date") \
    .withColumnRenamed("PreferredPaymentMethodID", "preferred_payment_method_id") \
    .select("customer_id", "full_name", "email", "phone_number",
            "date_of_birth", "registration_date", "preferred_payment_method_id")

processed_df.display()

## Step 5 — Load into Silver: SCD2 (Close Old Version + Insert New)

**5a.** For any `customer_id` already `is_current = true` in Silver where the incoming `email` differs, flip `is_current` to `false` and stamp `effective_end_date`. This `MERGE` only ever updates, never inserts.

**5b.** Insert the new version — every row in `processed_df` becomes a fresh Silver row with a new `customer_sk`, `is_current = true`.

In [ ]:
# 5a — close out old versions where email actually changed
silver_table = DeltaTable.forName(spark, SILVER_TABLE)

(silver_table.alias("tgt")
    .merge(
        processed_df.alias("src"),
        "tgt.customer_id = src.customer_id AND tgt.is_current = true "
        "AND tgt.email <> src.email"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "effective_end_date": "current_date()"
    })
    .execute()
)

print("Step 5a complete — old versions closed out where email changed")

In [ ]:
# 5b — insert new versions (covers both a changed customer and, if present, a brand-new one)
new_versions_df = processed_df \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("customer_sk",
        sha2(concat_ws("|", col("customer_id"), col("effective_start_date").cast("string")), 256)
    ) \
    .select("customer_sk", "customer_id", "full_name", "email", "phone_number",
            "date_of_birth", "registration_date", "preferred_payment_method_id",
            "is_current", "effective_start_date", "effective_end_date")

new_versions_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)
print(f"Step 5b complete — {new_versions_df.count()} new version row(s) inserted")

## Step 6 — Verify

Each changed `customer_id` should now show **2 rows**: old version (`is_current=false`, `effective_end_date` set) and new version (`is_current=true`, `effective_end_date=NULL`).

In [ ]:
df = spark.table(SILVER_TABLE)
print(f"{SILVER_TABLE} rows : {df.count():,}")

df.filter(col("customer_id").isin(sample_ids)) \
  .select("customer_sk", "customer_id", "email", "is_current",
          "effective_start_date", "effective_end_date") \
  .orderBy("customer_id", "effective_start_date") \
  .display()

**Q: For your 2 changed customers, confirm you see exactly 2 rows each. What are the `effective_start_date`/`effective_end_date` values on the old vs. new row?**

*Your answer:* _______________

## Submission Checklist

```
☐ Scratch schema created, bronze_customers + silver_customers cloned
☐ CDF enabled on the cloned bronze table
☐ 2 customer emails changed and confirmed via Step 3's CDF read
☐ Step 5a/5b MERGE executed without error
☐ Step 6 verified — exactly 2 rows per changed customer_id
```

**What's next:** HOL 2 takes this exact MERGE technique one level up — from a dimension to the fact table itself.